# División estratégica de la base de mora bancaria

Este notebook divide la base sintética de 21.500 clientes en tres conjuntos independientes:

- **Entrenamiento:** 70 % (15.050 registros).
- **Validación:** 15 % (3.225 registros).
- **Prueba:** 15 % (3.225 registros).

La división es **estratificada por la variable objetivo `mora`**, por lo que conserva la representación de sus seis clases. Además, el notebook verifica la presencia de todas las categorías bancarias relevantes, los patrones de datos faltantes, la exclusividad de los identificadores y la ausencia de pérdida de registros.

No se imputan, normalizan ni transforman variables antes de la división, para prevenir fuga de información entre los conjuntos.


## 1. Importación de librerías y parámetros

Se fijan la semilla, los nombres de archivo y los tamaños exactos para que el proceso sea reproducible tanto en Google Colab como en un entorno local.


In [1]:
# Importa Path para manejar rutas de archivos de forma compatible entre sistemas.
from pathlib import Path
# Importa pandas para leer, dividir, validar y exportar los datos tabulares.
import pandas as pd
# Importa la función que realiza particiones aleatorias con estratificación.
from sklearn.model_selection import train_test_split
# Fija una semilla única para reproducir exactamente la misma división en cada ejecución.
SEMILLA = 20260903
# Define el nombre esperado del archivo completo que se cargará en el notebook.
ARCHIVO_ORIGEN = Path("dataset_mora.csv")
# Define el archivo que almacenará el set de entrenamiento.
ARCHIVO_TRAIN = Path("data_train-mora.csv")
# Define el archivo que almacenará el set de validación.
ARCHIVO_VALIDATION = Path("data_validation-mora.csv")
# Define el archivo que almacenará el set de prueba.
ARCHIVO_TEST = Path("data_test-mora.csv")
# Define la columna objetivo utilizada para la estratificación.
COLUMNA_OBJETIVO = "mora"
# Establece la cantidad total de filas que debe contener la base original.
TOTAL_ESPERADO = 21500
# Establece la cantidad exacta de filas del conjunto de entrenamiento.
FILAS_TRAIN = 15050
# Establece la cantidad exacta de filas del conjunto de validación.
FILAS_VALIDATION = 3225
# Establece la cantidad exacta de filas del conjunto de prueba.
FILAS_TEST = 3225
# Enumera las variables categóricas cuya cobertura se comprobará en cada conjunto.
COLUMNAS_CATEGORICAS_CONTROL = ["mora", "zona", "tipo_credito", "vivienda", "nivel_ahorro", "clasif_sbs", "nivel_educ", "reestructuraciones_refinanciamientos_previos"]
# Muestra los parámetros principales para dejar una evidencia visible de la configuración.
print(f"Semilla reproducible: {SEMILLA}")
# Muestra los tamaños exactos que se generarán.
print(f"Tamaños esperados: train={FILAS_TRAIN}, validation={FILAS_VALIDATION}, test={FILAS_TEST}")


Semilla reproducible: 20260903
Tamaños esperados: train=15050, validation=3225, test=3225


## 2. Carga del archivo en Google Colab

Si el CSV no se encuentra en la carpeta de trabajo, se abre automáticamente el selector de archivos de Colab. Se debe cargar `dataset_mora.csv`.


In [2]:
# Comprueba si la base completa ya está disponible en el entorno de ejecución.
if not ARCHIVO_ORIGEN.exists():
    # Intenta importar el cargador interactivo disponible exclusivamente en Google Colab.
    try:
        # Importa la utilidad de archivos para solicitar la carga manual desde el navegador.
        from google.colab import files
    # Captura el caso en que el notebook se ejecuta fuera de Google Colab.
    except ImportError as error:
        # Informa de forma explícita qué archivo debe copiarse junto al notebook.
        raise FileNotFoundError(f"No se encontró {ARCHIVO_ORIGEN.name}. Copie el CSV en la carpeta del notebook y vuelva a ejecutar.") from error
    # Abre el selector de archivos de Google Colab y conserva la lista de archivos recibidos.
    archivos_subidos = files.upload()
    # Verifica que el usuario haya seleccionado el archivo con el nombre esperado.
    if ARCHIVO_ORIGEN.name not in archivos_subidos:
        # Detiene la ejecución si el archivo solicitado no fue cargado.
        raise FileNotFoundError(f"Debe cargar el archivo con el nombre exacto: {ARCHIVO_ORIGEN.name}")
# Confirma la ruta que se utilizará para leer la base completa.
print(f"Archivo de origen localizado: {ARCHIVO_ORIGEN.resolve()}")


Archivo de origen localizado: /content/dataset_mora.csv


## 3. Lectura y validación inicial

La cédula se lee como texto para conservar ceros iniciales. Antes de dividir, se verifica el tamaño, la estructura, la variable objetivo y la unicidad de los identificadores.


In [3]:
# Lee el CSV y fuerza el identificador a texto para preservar sus diez dígitos.
datos = pd.read_csv(ARCHIVO_ORIGEN, dtype={"ID": "string"})
# Verifica que la base contenga exactamente el número de registros solicitado.
if len(datos) != TOTAL_ESPERADO:
    # Detiene el proceso si el tamaño de la base no coincide con 21.500 filas.
    raise ValueError(f"La base contiene {len(datos)} filas y se esperaban {TOTAL_ESPERADO}.")
# Comprueba que exista la variable objetivo utilizada para la predicción.
if COLUMNA_OBJETIVO not in datos.columns:
    # Detiene el proceso si no existe la columna mora.
    raise KeyError(f"No existe la columna objetivo {COLUMNA_OBJETIVO}.")
# Comprueba que la variable mora sea la última columna de la base.
if datos.columns[-1] != COLUMNA_OBJETIVO:
    # Detiene el proceso si la posición de la variable objetivo no es la acordada.
    raise ValueError("La columna mora debe encontrarse al final de la base.")
# Comprueba que no existan identificadores vacíos.
if datos["ID"].isna().any():
    # Detiene el proceso porque cada cliente debe tener un identificador completo.
    raise ValueError("La columna ID contiene valores faltantes.")
# Comprueba que cada cliente tenga un identificador único.
if not datos["ID"].is_unique:
    # Detiene el proceso para evitar que el mismo cliente aparezca en más de una partición.
    raise ValueError("La columna ID contiene valores duplicados.")
# Comprueba que la variable objetivo esté completa antes de estratificar.
if datos[COLUMNA_OBJETIVO].isna().any():
    # Detiene el proceso porque la estratificación requiere una clase de mora para cada fila.
    raise ValueError("La columna mora contiene valores faltantes.")
# Comprueba que todas las variables categóricas de control existan en la base.
if not set(COLUMNAS_CATEGORICAS_CONTROL).issubset(datos.columns):
    # Detiene el proceso si falta alguna variable necesaria para comprobar la cobertura.
    raise KeyError("Falta una o más columnas categóricas requeridas para el control de cobertura.")
# Informa el tamaño confirmado de la base antes de realizar la división.
print(f"Base validada: {datos.shape[0]} filas y {datos.shape[1]} columnas.")
# Muestra la cantidad de clases disponibles en la variable objetivo.
print(f"Clases de mora disponibles: {datos[COLUMNA_OBJETIVO].nunique()}.")


Base validada: 21500 filas y 34 columnas.
Clases de mora disponibles: 6.


## 4. División estratificada 70 % / 15 % / 15 %

Primero se separa el 70 % para entrenamiento. El 30 % restante vuelve a dividirse en partes iguales para validación y prueba. En ambos pasos se usa `mora` como estrato.


In [4]:
# Separa exactamente 15.050 registros para entrenamiento y reserva los 6.450 restantes.
data_train, datos_temporales = train_test_split(datos, train_size=FILAS_TRAIN, random_state=SEMILLA, shuffle=True, stratify=datos[COLUMNA_OBJETIVO])
# Divide la reserva en exactamente 3.225 registros de validación y 3.225 registros de prueba.
data_validation, data_test = train_test_split(datos_temporales, train_size=FILAS_VALIDATION, test_size=FILAS_TEST, random_state=SEMILLA + 1, shuffle=True, stratify=datos_temporales[COLUMNA_OBJETIVO])
# Reinicia el índice del conjunto de entrenamiento sin convertirlo en una nueva columna.
data_train = data_train.reset_index(drop=True)
# Reinicia el índice del conjunto de validación sin convertirlo en una nueva columna.
data_validation = data_validation.reset_index(drop=True)
# Reinicia el índice del conjunto de prueba sin convertirlo en una nueva columna.
data_test = data_test.reset_index(drop=True)
# Verifica el tamaño exacto obtenido para el conjunto de entrenamiento.
if len(data_train) != FILAS_TRAIN:
    # Detiene el proceso si la proporción de entrenamiento no produjo 15.050 filas.
    raise AssertionError(f"Train contiene {len(data_train)} filas y debe contener {FILAS_TRAIN}.")
# Verifica el tamaño exacto obtenido para el conjunto de validación.
if len(data_validation) != FILAS_VALIDATION:
    # Detiene el proceso si la proporción de validación no produjo 3.225 filas.
    raise AssertionError(f"Validation contiene {len(data_validation)} filas y debe contener {FILAS_VALIDATION}.")
# Verifica el tamaño exacto obtenido para el conjunto de prueba.
if len(data_test) != FILAS_TEST:
    # Detiene el proceso si la proporción de prueba no produjo 3.225 filas.
    raise AssertionError(f"Test contiene {len(data_test)} filas y debe contener {FILAS_TEST}.")
# Confirma en pantalla los tamaños finales de los tres conjuntos.
print(f"División creada: train={len(data_train)}, validation={len(data_validation)}, test={len(data_test)}")


División creada: train=15050, validation=3225, test=3225


## 5. Controles de cobertura, coherencia y ausencia de fuga

Se verifica que:

- Los tres conjuntos tengan el mismo esquema de 34 variables.
- Todas las clases de `mora` estén presentes.
- Todas las categorías de las variables bancarias seleccionadas estén representadas.
- Las columnas que contienen faltantes en la base también tengan ejemplos faltantes en cada conjunto.
- Ningún `ID` se repita entre entrenamiento, validación y prueba.
- La unión de los tres archivos reconstruya exactamente los 21.500 clientes.
- El valor de `atraso`, cuando está disponible, sea coherente con el texto de `mora`.


In [5]:
# Define una función que comprueba que el número de atraso coincida con la clase textual de mora.
def validar_atraso_mora(conjunto):
    # Crea una máscara para los clientes al día que tengan el número de atraso informado.
    al_dia = conjunto[COLUMNA_OBJETIVO].eq("AL DÍA") & conjunto["atraso"].notna()
    # Crea una máscara para la mora de uno a siete días con atraso informado.
    mora_1_7 = conjunto[COLUMNA_OBJETIVO].eq("MORA DE 1 A 7 DÍAS") & conjunto["atraso"].notna()
    # Crea una máscara para la mora de ocho a treinta días con atraso informado.
    mora_8_30 = conjunto[COLUMNA_OBJETIVO].eq("MORA DE 8 A 30 DÍAS") & conjunto["atraso"].notna()
    # Crea una máscara para la mora de treinta y uno a sesenta días con atraso informado.
    mora_31_60 = conjunto[COLUMNA_OBJETIVO].eq("MORA DE 31 A 60 DÍAS") & conjunto["atraso"].notna()
    # Crea una máscara para la mora de sesenta y uno a noventa días con atraso informado.
    mora_61_90 = conjunto[COLUMNA_OBJETIVO].eq("MORA DE 61 A 90 DÍAS") & conjunto["atraso"].notna()
    # Crea una máscara para la mora mayor a noventa días con atraso informado.
    mora_mayor_90 = conjunto[COLUMNA_OBJETIVO].eq("MORA DE MÁS DE 90 DÍAS") & conjunto["atraso"].notna()
    # Comprueba que los clientes al día tengan cero días de atraso cuando el valor está disponible.
    regla_al_dia = conjunto.loc[al_dia, "atraso"].eq(0).all()
    # Comprueba que la mora leve esté entre uno y siete días.
    regla_1_7 = conjunto.loc[mora_1_7, "atraso"].between(1, 7).all()
    # Comprueba que la mora de corto plazo esté entre ocho y treinta días.
    regla_8_30 = conjunto.loc[mora_8_30, "atraso"].between(8, 30).all()
    # Comprueba que la mora media esté entre treinta y uno y sesenta días.
    regla_31_60 = conjunto.loc[mora_31_60, "atraso"].between(31, 60).all()
    # Comprueba que la mora alta esté entre sesenta y uno y noventa días.
    regla_61_90 = conjunto.loc[mora_61_90, "atraso"].between(61, 90).all()
    # Comprueba que la mora crítica sea superior a noventa días.
    regla_mayor_90 = conjunto.loc[mora_mayor_90, "atraso"].gt(90).all()
    # Devuelve verdadero únicamente cuando todas las reglas de coherencia se cumplen.
    return all([regla_al_dia, regla_1_7, regla_8_30, regla_31_60, regla_61_90, regla_mayor_90])
# Define una función general para validar la representatividad de cada subconjunto.
def validar_conjunto(nombre, conjunto, base):
    # Comprueba que el orden y los nombres de las columnas sean iguales a los de la base original.
    if list(conjunto.columns) != list(base.columns):
        # Detiene el proceso si la estructura de variables cambió durante la división.
        raise AssertionError(f"{nombre} no conserva el esquema original.")
    # Obtiene todas las clases de mora de la base completa.
    clases_base = set(base[COLUMNA_OBJETIVO].dropna().unique())
    # Obtiene todas las clases de mora presentes en el subconjunto evaluado.
    clases_conjunto = set(conjunto[COLUMNA_OBJETIVO].dropna().unique())
    # Comprueba que el subconjunto contenga todas las clases objetivo.
    if clases_conjunto != clases_base:
        # Detiene el proceso si falta alguna clase necesaria para el modelado.
        raise AssertionError(f"{nombre} no contiene todas las clases de mora.")
    # Recorre las variables categóricas seleccionadas para comprobar su cobertura.
    for columna in COLUMNAS_CATEGORICAS_CONTROL:
        # Obtiene las categorías no nulas disponibles en la base completa.
        categorias_base = set(base[columna].dropna().unique())
        # Obtiene las categorías no nulas presentes en el subconjunto evaluado.
        categorias_conjunto = set(conjunto[columna].dropna().unique())
        # Comprueba que el subconjunto contenga exactamente las categorías de la base.
        if categorias_conjunto != categorias_base:
            # Detiene el proceso e identifica la columna cuya cobertura no es completa.
            raise AssertionError(f"{nombre} no conserva todas las categorías de {columna}.")
    # Identifica las columnas que tienen al menos un valor faltante en la base completa.
    columnas_con_faltantes = base.columns[base.isna().any()].tolist()
    # Busca columnas cuyo patrón faltante no haya quedado representado en el subconjunto.
    faltantes_sin_muestra = [columna for columna in columnas_con_faltantes if not conjunto[columna].isna().any()]
    # Comprueba que cada variable incompleta tenga ejemplos faltantes en el subconjunto.
    if faltantes_sin_muestra:
        # Detiene el proceso y enumera las variables sin representación de valores faltantes.
        raise AssertionError(f"{nombre} no contiene faltantes de: {faltantes_sin_muestra}")
    # Comprueba la relación lógica entre el atraso numérico y la clase textual de mora.
    if not validar_atraso_mora(conjunto):
        # Detiene el proceso si encuentra una clasificación de mora incompatible con los días de atraso.
        raise AssertionError(f"{nombre} contiene inconsistencias entre atraso y mora.")
    # Devuelve una confirmación legible cuando todas las comprobaciones concluyen correctamente.
    return f"{nombre}: cobertura completa y coherencia aprobada"
# Ejecuta los controles de cobertura y coherencia sobre el conjunto de entrenamiento.
print(validar_conjunto("TRAIN", data_train, datos))
# Ejecuta los controles de cobertura y coherencia sobre el conjunto de validación.
print(validar_conjunto("VALIDATION", data_validation, datos))
# Ejecuta los controles de cobertura y coherencia sobre el conjunto de prueba.
print(validar_conjunto("TEST", data_test, datos))
# Convierte los identificadores de entrenamiento en un conjunto para comparar su exclusividad.
ids_train = set(data_train["ID"])
# Convierte los identificadores de validación en un conjunto para comparar su exclusividad.
ids_validation = set(data_validation["ID"])
# Convierte los identificadores de prueba en un conjunto para comparar su exclusividad.
ids_test = set(data_test["ID"])
# Comprueba que no existan clientes compartidos entre entrenamiento y validación.
if ids_train & ids_validation:
    # Detiene el proceso si detecta fuga de clientes entre esos dos conjuntos.
    raise AssertionError("Existen IDs compartidos entre train y validation.")
# Comprueba que no existan clientes compartidos entre entrenamiento y prueba.
if ids_train & ids_test:
    # Detiene el proceso si detecta fuga de clientes entre esos dos conjuntos.
    raise AssertionError("Existen IDs compartidos entre train y test.")
# Comprueba que no existan clientes compartidos entre validación y prueba.
if ids_validation & ids_test:
    # Detiene el proceso si detecta fuga de clientes entre esos dos conjuntos.
    raise AssertionError("Existen IDs compartidos entre validation y test.")
# Une los identificadores de los tres conjuntos para comprobar que no se perdió ningún cliente.
ids_reconstruidos = ids_train | ids_validation | ids_test
# Compara la unión de las particiones contra todos los identificadores originales.
if ids_reconstruidos != set(datos["ID"]):
    # Detiene el proceso si falta o sobra algún identificador después de la división.
    raise AssertionError("La unión de los conjuntos no reconstruye la base original.")
# Confirma que los clientes quedaron separados sin cruces y sin pérdida de filas.
print("IDs exclusivos y cobertura total de los 21.500 clientes: control aprobado.")


TRAIN: cobertura completa y coherencia aprobada
VALIDATION: cobertura completa y coherencia aprobada
TEST: cobertura completa y coherencia aprobada
IDs exclusivos y cobertura total de los 21.500 clientes: control aprobado.


## 6. Comparación de la distribución de `mora`

La siguiente tabla permite revisar que la proporción de cada clase objetivo sea prácticamente igual en la base completa y en los tres conjuntos.


In [6]:
# Calcula el porcentaje de cada clase de mora en la base completa.
porcentaje_total = datos[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index()
# Calcula el porcentaje de cada clase de mora en el conjunto de entrenamiento.
porcentaje_train = data_train[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index()
# Calcula el porcentaje de cada clase de mora en el conjunto de validación.
porcentaje_validation = data_validation[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index()
# Calcula el porcentaje de cada clase de mora en el conjunto de prueba.
porcentaje_test = data_test[COLUMNA_OBJETIVO].value_counts(normalize=True).mul(100).sort_index()
# Reúne los cuatro porcentajes en una tabla comparativa y redondea a cuatro decimales.
distribucion_mora = pd.DataFrame({"base_total_pct": porcentaje_total, "train_pct": porcentaje_train, "validation_pct": porcentaje_validation, "test_pct": porcentaje_test}).round(4)
# Calcula la mayor diferencia absoluta en puntos porcentuales frente a la base completa.
desviacion_maxima = distribucion_mora[["train_pct", "validation_pct", "test_pct"]].sub(distribucion_mora["base_total_pct"], axis=0).abs().to_numpy().max()
# Comprueba que la estratificación mantenga una desviación máxima inferior a 0,05 puntos porcentuales.
if desviacion_maxima > 0.05:
    # Detiene el proceso si la distribución de la variable objetivo se aleja del límite definido.
    raise AssertionError(f"La desviación máxima de mora es {desviacion_maxima:.4f} puntos porcentuales.")
# Muestra la tabla de distribución para documentar el equilibrio de las particiones.
print(distribucion_mora.to_string())
# Muestra la desviación máxima observada entre los subconjuntos y la base completa.
print(f"Desviación máxima respecto de la base: {desviacion_maxima:.4f} puntos porcentuales.")


                        base_total_pct  train_pct  validation_pct  test_pct
mora                                                                       
AL DÍA                         72.0047    72.0066         72.0000   72.0000
MORA DE 1 A 7 DÍAS             10.0000    10.0000         10.0155    9.9845
MORA DE 31 A 60 DÍAS            4.4977     4.4983          4.4961    4.4961
MORA DE 61 A 90 DÍAS            3.4977     3.4950          3.5039    3.5039
MORA DE 8 A 30 DÍAS             7.0000     7.0033          6.9767    7.0078
MORA DE MÁS DE 90 DÍAS          3.0000     2.9967          3.0078    3.0078
Desviación máxima respecto de la base: 0.0233 puntos porcentuales.


## 7. Exportación de los tres archivos CSV

Los archivos se guardan en UTF-8 con BOM (`utf-8-sig`) para facilitar su apertura en Excel sin perder tildes ni caracteres especiales. El índice interno de pandas no se exporta.


In [7]:
# Exporta el conjunto de entrenamiento sin agregar una columna de índice.
data_train.to_csv(ARCHIVO_TRAIN, index=False, encoding="utf-8-sig")
# Exporta el conjunto de validación sin agregar una columna de índice.
data_validation.to_csv(ARCHIVO_VALIDATION, index=False, encoding="utf-8-sig")
# Exporta el conjunto de prueba sin agregar una columna de índice.
data_test.to_csv(ARCHIVO_TEST, index=False, encoding="utf-8-sig")
# Confirma la creación del archivo de entrenamiento y su número de filas.
print(f"Creado: {ARCHIVO_TRAIN.name} | {len(data_train)} filas")
# Confirma la creación del archivo de validación y su número de filas.
print(f"Creado: {ARCHIVO_VALIDATION.name} | {len(data_validation)} filas")
# Confirma la creación del archivo de prueba y su número de filas.
print(f"Creado: {ARCHIVO_TEST.name} | {len(data_test)} filas")


Creado: data_train-mora.csv | 15050 filas
Creado: data_validation-mora.csv | 3225 filas
Creado: data_test-mora.csv | 3225 filas


## 8. Descarga automática en Google Colab

En Colab, esta celda inicia la descarga de los tres CSV. En un entorno local, los archivos permanecen en la carpeta del notebook.


In [8]:
# Intenta importar el módulo de descarga disponible en Google Colab.
try:
    # Importa la utilidad de archivos para descargar los resultados al computador.
    from google.colab import files
# Captura el caso en que la ejecución se realice fuera de Google Colab.
except ImportError:
    # Informa que los archivos ya quedaron guardados localmente.
    print("Ejecución local: los tres CSV quedaron guardados junto al notebook.")
# Ejecuta este bloque únicamente cuando Google Colab está disponible.
else:
    # Inicia la descarga del archivo de entrenamiento.
    files.download(str(ARCHIVO_TRAIN))
    # Inicia la descarga del archivo de validación.
    files.download(str(ARCHIVO_VALIDATION))
    # Inicia la descarga del archivo de prueba.
    files.download(str(ARCHIVO_TEST))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>